# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Ruta de esta práctica · ML · Encuentro 3: reducción de dimensiones
Variables originales → estandarización → componentes → varianza conservada. PCA no recibe etiquetas de una tarea de predicción.

**Antes de comenzar:** guarda tu copia personal. Usa el índice lateral de Colab para localizar el bloque indicado. Ejecuta las celdas de preparación y después el bloque, en orden; cambiar una variable exige volver a ejecutar su celda y las que dependen de ella. Abrir una figura no vuelve a entrenar un modelo.

**Entorno:** estos ejemplos están preparados para CPU. No requieren claves de API, contratar servicios ni activar un entorno de pago. La disponibilidad y límites de los recursos de Colab dependen de tu cuenta; no cambies a recursos de pago para esta práctica. Si se interrumpe la sesión, conserva el cuaderno y vuelve a ejecutar desde la preparación.

**Distingue siempre:** datos observados o simulados → parámetros aprendidos por `fit`/entrenamiento → configuraciones que tú eliges → resultados en datos reservados. Una buena cifra de entrenamiento no demuestra aprendizaje útil en nuevos casos.


## PCA: de tres variables a dos componentes
Datos sintéticos X1, X2 y X3. Tras escalar, PCA combina variables. Lee los porcentajes calculados en esta ejecución: no supongas 70 % o 90 % fijos. Varianza capturada no equivale a accuracy ni garantiza conservar lo relevante para una tarea.

### Construir una componente y entender lo que pierde
Las tres variables no tienen unidades físicas definidas: son números simulados. Primero calculamos zⱼ=(Xⱼ−μⱼ)/σⱼ. PCA centra esas coordenadas y aprende direcciones ortogonales que capturan dispersión. Una componente se calcula con un producto punto de la fila centrada y sus pesos; la celda adicional muestra una fila completa, sin redondear durante el cálculo.

**Varianza explicada** divide la varianza de cada componente por la suma de todas. Sumar PC1 y PC2 indica lo conservado al dibujar en 2D; lo restante se descarta. El eje vertical de la vista 1D está puesto a cero para dibujar: no es otra variable medida. Las direcciones pueden invertir su signo entre versiones sin cambiar la representación geométrica ni la varianza conservada.

In [ ]:
# ================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA (Colab listo para pegar)
# Caso simple: dataset sintético de 3 variables
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset con 3 variables correlacionadas.
# 2) Visualiza los datos en 3D.
# 3) Aplica PCA para reducir a 2D y a 1D.
# 4) Explica cómo leer los resultados: varianza explicada, componentes, proyecciones.
#
# ------------------------------------------------
# ¿Qué es PCA?
# ------------------------------------------------
# PCA = Principal Component Analysis.
# Objetivo: encontrar nuevas variables (componentes principales) que:
# - Son combinaciones lineales de las originales.
# - La primera maximiza varianza; las siguientes son ortogonales y capturan la restante.
# - Para un número fijo de dimensiones, minimizan el error de reconstrucción cuadrático.
# - La varianza conservada no garantiza conservar lo relevante para otra tarea.
#
# Ejemplo:
# - Si lluvia y humedad están muy correlacionadas, PCA crea una sola
#   componente que resume ambas, reduciendo redundancia.
# ================================================================

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D

# ----------------------------
# 1) Dataset sintético
# ----------------------------
# Creamos 3 variables con correlaciones:
# - X1 ~ normal
# - X2 ~ X1 con ruido (correlación alta)
# - X3 ~ independiente
rng = np.random.default_rng(42)
n = 200
X1 = rng.normal(50, 10, n)
X2 = X1 * 0.8 + rng.normal(0, 5, n) + 10
X3 = rng.normal(30, 8, n)

df = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})
display(df.head())

# ----------------------------
# 2) Visualización en 3D
# ----------------------------
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df["X1"], df["X2"], df["X3"], alpha=0.6)
ax.set_xlabel("X1")
ax.set_ylabel("X2")
ax.set_zlabel("X3")
ax.set_title("Datos originales en 3D")
plt.show()

# ----------------------------
# 3) Escalado
# ----------------------------
# PCA es sensible a escala → StandardScaler.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 4) PCA
# ----------------------------
pca = PCA(n_components=3)  # tantas como originales para analizar
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"Componente {i+1}: {v:.2%}")

print("\nVarianza acumulada:", np.cumsum(pca.explained_variance_ratio_))

print("\nComponentes (pesos aplicados a cada variable estandarizada en cada PC):")
print(pd.DataFrame(pca.components_, columns=df.columns, index=[f"PC{i+1}" for i in range(3)]))

# ----------------------------
# 5) Visualización PCA 2D
# ----------------------------
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Datos reducidos a 2D con PCA")
plt.grid(True)
plt.show()

# ----------------------------
# 6) Visualización PCA 1D
# ----------------------------
plt.figure(figsize=(6,2))
plt.scatter(X_pca[:,0], np.zeros_like(X_pca[:,0]), alpha=0.6)
plt.xlabel("PC1 (única dimensión)")
plt.yticks([])
plt.title("Datos reducidos a 1D con PCA")
plt.show()

# ----------------------------
# 7) Interpretación
# ----------------------------
print(f"""
================= CÓMO LEER LOS RESULTADOS =================
1) Varianza explicada:
   - PC1 captura la mayor proporción de varianza; consulta el porcentaje calculado.
   - PC2 añade otra parte significativa.
   - PC3 suele explicar poco si hay redundancia entre variables.
   → Decisión: consulta la varianza acumulada calculada para decidir cuántas componentes conservar.

2) Componentes (matriz de pesos):
   - Cada PC combina z1, z2, z3: zj = (Xj - mediaj) / desviaciónj.
   - PC1 = w1*z1 + w2*z2 + w3*z3; los pesos de la tabla actúan tras escalar.
   - Si PC1 tiene coeficientes altos y parecidos en X1 y X2,
     significa que resume la "información común" entre ambas.

3) Gráfica 2D:
   - PC1 y PC2 conservan {pca.explained_variance_ratio_[:2].sum():.2%} de la varianza estandarizada.
   - Se descarta {1-pca.explained_variance_ratio_[:2].sum():.2%}; no son las distancias originales sin escalar.

4) Gráfica 1D:
   - PC1 conserva {pca.explained_variance_ratio_[0]:.2%} y descarta {1-pca.explained_variance_ratio_[0]:.2%} de varianza.
   - Una sola dimensión puede perder estructura relevante: depende de la tarea.

5) Uso práctico:
   - PCA reduce variables redundantes.
   - Puede ayudar a visualizar y preparar datos para otros algoritmos.
   - Una mejora de clustering o regresión requiere comparación; no está garantizada.
=============================================================
""")


### Una fila, una componente calculada paso a paso
La fila original se estandariza con las estadísticas ya aprendidas. Después restamos el pequeño centrado de PCA y multiplicamos por los pesos de PC1. La suma debe coincidir con la primera coordenada del gráfico.

In [ ]:
fila_original = df.iloc[[0]]
fila_z = scaler.transform(fila_original)[0]
fila_centrada = fila_z - pca.mean_
productos = fila_centrada * pca.components_[0]
display(pd.DataFrame({'Variable':df.columns,'Original':fila_original.iloc[0].values,
                      'Media':scaler.mean_,'Desviación':scaler.scale_,
                      'z centrado':fila_centrada,'Peso PC1':pca.components_[0],
                      'Producto':productos}))
print('PC1 = suma de productos =',productos.sum(),'· coordenada del gráfico =',X_pca[0,0])
assert np.isclose(productos.sum(),X_pca[0,0])
